# 06 · Ehrenfest-Modell und Entropie

Die Aufgaben sind in einzelne Stationen eingeteilt. Lies jeweils zuerst die Aufgabenstellung und mach dir eigene Gedanken und probiere dann das jeweilige Widget aus. Speichere deine Codeänderungen mit **Strg + S**.

---

Es gibt **N unterscheidbare Moleküle in zwei Behältern**. In jedem Schritt wird eine von **N + 1 gleich wahrscheinlichen Aktionen** gewählt: eines der Moleküle wechselt den Behälter, oder es geschieht nichts. $k$ bezeichnet die Anzahl in Behälter 0.

**Schwierigkeitsstufen**

| Stufe | Orientierung |
|---|---|
| Leicht | Ab Klasse 9: Wahrscheinlichkeiten, Baumdiagramme, Bruchrechnen. Oft genügen Ausprobieren und genaues Lesen. |
| Schwerer | Ab Klasse 10/11: Matrizen, Gleichungssysteme, mehrstufige Überlegungen und Argumentieren mit Begriffen aus dem Skript. |
| Schwer | Eher Klasse 12/13: Knobelaufgaben, anspruchsvollere Begründungen, Beweisideen und ungewohnte Modellierung. |

Die Klassenstufen sind eine Orientierung, keine Zugangsvoraussetzung. Du kannst schwierigere Teilaufgaben auch überspringen, die Reihenfolge ist nicht obligatorisch.

<br>

## Vorbereitung der Widgets
#### Klicke zuerst oben im Menü auf **Run → Run All Cells**. Den Code in der hier folgenden Zelle brauchst du nicht zu bearbeiten.

In [ ]:
# Pakete und Widgets vorbereiten.
import sys

try:
    import ipywidgets as W
except ModuleNotFoundError:
    if sys.platform == "emscripten":
        import piplite

        await piplite.install("ipywidgets>=8,<9")
        import ipywidgets as W
    else:
        raise RuntimeError("Bitte ipywidgets in dieser Python-Umgebung installieren lassen.")


import io
import math
import html
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

PANELS = {}


# Ehrenfest-Modell und gemeinsame Widget-Hilfen.
def matrix(N):
    M = np.zeros((N + 1, N + 1))
    for k in range(N + 1):
        M[k, k] = 1 / (N + 1)
        if k > 0:
            M[k - 1, k] = k / (N + 1)
        if k < N:
            M[k + 1, k] = (N - k) / (N + 1)
    return M


def stationaer(N):
    return np.array([math.comb(N, k) / 2**N for k in range(N + 1)])


def entropien(N):
    return np.log(np.array([float(math.comb(N, k)) for k in range(N + 1)]))


def mikropfad(N, schritte, k0, seed):
    rng = np.random.default_rng(seed)
    actions = rng.integers(0, N + 1, size=schritte)
    states = np.zeros((schritte + 1, N), dtype=bool)
    states[0, :k0] = True
    for t, a in enumerate(actions):
        states[t + 1] = states[t]
        if a < N:
            states[t + 1, a] = not states[t + 1, a]
    return {"states": states, "actions": actions, "k": states.sum(axis=1)}


def makropfade(N, schritte, k0, seed, anzahl=100):
    rng = np.random.default_rng(seed)
    x = np.empty((anzahl, schritte + 1), dtype=np.int16)
    x[:, 0] = k0
    for t in range(schritte):
        a = rng.integers(0, N + 1, size=anzahl)
        k = x[:, t]
        x[:, t + 1] = k + np.where(a < k, -1, np.where(a < N, 1, 0))
    return x


def rueckkehrzeiten(N, anzahl, grenze, seed):
    rng = np.random.default_rng(seed)
    k = np.full(anzahl, N)
    times = np.full(anzahl, np.inf)
    for t in range(1, grenze + 1):
        active = np.flatnonzero(np.isinf(times))
        if len(active) == 0:
            break
        old = k[active]
        a = rng.integers(0, N + 1, size=len(active))
        k[active] = old + np.where(a < old, -1, np.where(a < N, 1, 0))
        times[active[k[active] == N]] = t
    return times


def regler(value, minimum, maximum, step=1):
    return W.IntSlider(
        value=value,
        min=minimum,
        max=maximum,
        step=step,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )


def auswahl(options, value):
    return W.Dropdown(options=options, value=value, layout=W.Layout(width="300px"))


def feld(label, w):
    return W.VBox([W.HTML(html.escape(label)), w])


def knopf(label="Neu auslosen"):
    return W.Button(description=label, button_style="info", layout=W.Layout(width="190px"))


def bild(width="850px"):
    return W.Image(format="png", layout=W.Layout(width=width, max_width="100%"))


def zeichnen(fig, w):
    fig.tight_layout()
    with io.BytesIO() as b:
        fig.savefig(b, format="png", dpi=105)
        w.value = b.getvalue()


def tabelle(head, rows):
    def cell(x, tag):
        return f'<{tag} style="padding:4px 9px;text-align:center">{html.escape(str(x))}</{tag}>'

    return (
        '<div style="overflow:auto"><table><tr>'
        + "".join(cell(v, "th") for v in head)
        + "</tr>"
        + "".join("<tr>" + "".join(cell(v, "td") for v in row) + "</tr>" for row in rows)
        + "</table></div>"
    )


# Stationen.
def station_teilchen():
    N = auswahl([4, 6, 8, 12], 6)
    t = regler(0, 0, 40)
    btn = knopf()
    pic = bild()
    info = W.HTML()
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.5))
    plt.close(fig)
    r = {"N": N, "t": t, "button": btn, "bild": pic, "seed": 2026, "key": None, "last": None}

    def update(_=None):
        key = (N.value, r["seed"])
        if r["key"] != key:
            r["data"] = mikropfad(N.value, 40, N.value, r["seed"])
            r["key"] = key
        state = r["data"]["states"][t.value]
        a = r["data"]["actions"][t.value - 1] if t.value else None
        for side, ax in enumerate(axes):
            ax.clear()
            ids = np.flatnonzero(state if side == 0 else ~state)
            ax.set(
                xlim=(-0.7, 3.7),
                ylim=(-2.7, 0.7),
                title=f"Behälter {side}: {len(ids)} Moleküle",
            )
            ax.set_xticks([])
            ax.set_yticks([])
            for i in ids:
                ax.scatter(
                    i % 4,
                    -(i // 4),
                    s=440,
                    color=("#287ab0" if side == 0 else "#d17a22"),
                    edgecolor=("crimson" if a == i else "white"),
                    linewidth=2,
                )
                ax.text(i % 4, -(i // 4), str(i + 1), color="white", ha="center", va="center")
        zeichnen(fig, pic)
        action = (
            "Start: alle Moleküle in Behälter 0."
            if a is None
            else (
                "Pause: kein Molekül wechselt."
                if a == N.value
                else f"Molekül {a+1} hat den Behälter gewechselt (rot umrandet)."
            )
        )
        info.value = f"<b>Schritt {t.value}: k = {int(state.sum())}</b><br>" + action
        r["last"] = {"state": state.copy(), "k": int(state.sum()), "action": a}

    def new(_):
        r["seed"] += 1
        update()

    N.observe(update, names="value")
    t.observe(update, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                feld("Anzahl Moleküle N", N),
                feld("Schritt der aufgezeichneten Folge", t),
                btn,
                pic,
                info,
            ]
        )
    )
    update()
    PANELS["teilchen"] = r
    return r


def station_uebergaenge():
    N = regler(4, 2, 8)
    k = regler(3, 0, 4)
    fields = [
        W.Text(placeholder="Zahl oder Bruch", layout=W.Layout(width="170px")) for _ in range(3)
    ]
    btn = knopf("Prüfen")
    info = W.HTML()
    mat = W.HTML()
    r = {"N": N, "k": k, "fields": fields, "button": btn, "last": None, "lock": False}

    def change(_=None):
        if r["lock"]:
            return
        r["lock"] = True
        k.max = N.value
        for w in fields:
            w.value = ""
        info.value = "Trage deine drei Wahrscheinlichkeiten ein."
        mat.value = ""
        r["last"] = None
        r["lock"] = False

    def edit(_):
        if not r["lock"]:
            r["last"] = None
            info.value = "Eingabe geändert; erneut prüfen."
            mat.value = ""

    def check(_):
        r["last"] = None
        try:
            p = np.array([float(Fraction(w.value.strip().replace(",", "."))) for w in fields])
        except (ValueError, ZeroDivisionError, OverflowError):
            info.value = "Bitte alle drei Wahrscheinlichkeiten eingeben."
            return
        target = np.array([k.value, 1, N.value - k.value]) / (N.value + 1)
        ok = np.all(np.isfinite(p)) and np.allclose(p, target, atol=1e-6, rtol=0)
        r["last"] = {"correct": bool(ok), "target": target, "M": matrix(N.value)}
        if not ok:
            info.value = "Zähle die Moleküle in jedem Behälter und die zusätzliche Pause. Alle N + 1 Aktionen sind gleich wahrscheinlich."
            return
        info.value = "<b>Richtig.</b> Hier ist die gesamte Matrix: Spalte = Zustand vorher, Zeile = Zustand danach. Prüfe auch die Randzustände."
        M = matrix(N.value)
        rows = []
        for i in range(N.value + 1):
            rows.append(
                [i] + [str(Fraction(float(v)).limit_denominator(N.value + 1)) for v in M[i]]
            )
        mat.value = tabelle(["danach ↓ / vorher →"] + list(range(N.value + 1)), rows)

    N.observe(change, names="value")
    k.observe(change, names="value")
    for w in fields:
        w.observe(edit, names="value")
    btn.on_click(check)
    display(
        W.VBox(
            [
                feld("N", N),
                feld("k: Moleküle in Behälter 0", k),
                W.HBox(
                    [
                        feld(label, w)
                        for label, w in zip(
                            ["k nimmt um 1 ab", "k bleibt gleich", "k nimmt um 1 zu"], fields
                        )
                    ]
                ),
                btn,
                info,
                mat,
            ]
        )
    )
    change()
    PANELS["uebergaenge"] = r
    return r


def station_gleichgewicht():
    N = auswahl([4, 10, 30, 100, 200, 1000], 10)
    k = regler(5, 0, 10)
    pic = bild()
    info = W.HTML()
    fig, ax = plt.subplots(figsize=(7.4, 3.3))
    plt.close(fig)
    r = {"N": N, "k": k, "bild": pic, "info": info, "last": None, "lock": False}

    def update(_=None):
        if r["lock"]:
            return
        n = N.value
        q = stationaer(n)
        j = k.value
        ks = np.arange(n + 1)
        nah = (5 * ks >= 2 * n) & (5 * ks <= 3 * n)
        ax.clear()
        ax.bar(ks / n, q, width=0.9 / n, color=np.where(nah, "#287ab0", "#b6cbd9"))
        ax.scatter([j / n], [q[j]], s=55, color="#c66b16", zorder=3)
        ax.set(
            xlim=(-0.03, 1.03),
            xlabel="Anteil k/N in Behälter 0",
            ylabel="Stationäre Wahrscheinlichkeit qₖ",
        )
        ax.grid(axis="y", alpha=0.2)
        zeichnen(fig, pic)
        count = math.comb(n, j)
        if count < 100000:
            count_text = str(count)
        else:
            mantisse, exponent = f"{count:.4e}".split("e")
            count_text = f"{mantisse} &middot; 10<sup>{int(exponent)}</sup>"
        info.value = f"k = {j}: <b>{count_text} Mikrozustände</b>; qₖ = {q[j]:.6g}.<br>"
        info.value += f"<b>Anteil zwischen 40 % und 60 % (dunkle Balken):</b> Wahrscheinlichkeit {q[nah].sum():.6g}.<br>"
        if j < n:
            a = q[j] * (n - j) / (n + 1)
            b = q[j + 1] * (j + 1) / (n + 1)
            info.value += f"Bei Startverteilung q: P(Xₜ = {j}, Xₜ₊₁ = {j+1}) = {a:.6g}; P(Xₜ = {j+1}, Xₜ₊₁ = {j}) = {b:.6g}."
        else:
            info.value += "Wähle k < N für den Vergleich der Übergänge k → k+1 und k+1 → k."
        r["last"] = {"q": q, "k": j, "count": count, "band_probability": float(q[nah].sum())}

    def newN(_):
        r["lock"] = True
        k.max = N.value
        k.value = N.value // 2
        r["lock"] = False
        update()

    N.observe(newN, names="value")
    k.observe(update, names="value")
    display(
        W.VBox(
            [feld("Anzahl Moleküle N", N), feld("Ausgewählter Makrozustand k", k), pic, info]
        )
    )
    update()
    PANELS["gleichgewicht"] = r
    return r


STARTS = ("Alle in Behälter 0", "Etwa halb verteilt", "Alle in Behälter 1")


def startwert(N, art):
    return N if art == STARTS[0] else (N // 2 if art == STARTS[1] else 0)


def station_relaxation():
    N = auswahl([10, 50, 200, 1000], 50)
    start = auswahl(STARTS, STARTS[0])
    dauer = regler(5, 1, 200)
    btn = knopf()
    pic = bild("1000px")
    info = W.HTML()
    fig, (ax, ah) = plt.subplots(
        1, 2, figsize=(9.5, 3.8), sharey=True, gridspec_kw={"width_ratios": [3, 1.25]}
    )
    plt.close(fig)
    r = {
        "N": N,
        "start": start,
        "dauer": dauer,
        "button": btn,
        "bild": pic,
        "seed": 2026,
        "key": None,
        "last": None,
    }

    def update(_=None):
        n = N.value
        T = dauer.value * n
        key = (n, start.value, r["seed"])
        if r["key"] != key:
            r["path"] = np.array([startwert(n, start.value)], dtype=np.int16)
            r["rng"] = np.random.default_rng(r["seed"])
            r["key"] = key
        # Nur neue Schritte berechnen; Zurückschieben und Verlängern behalten den bisherigen Verlauf.
        old = len(r["path"]) - 1
        if T > old:
            path = np.empty(T + 1, dtype=np.int16)
            path[: old + 1] = r["path"]
            actions = r["rng"].integers(0, n + 1, size=T - old)
            k = int(path[old])
            for t, a in enumerate(actions, start=old):
                k += -1 if a < k else (1 if a < n else 0)
                path[t + 1] = k
            r["path"] = path
        x = r["path"][: T + 1]
        ax.clear()
        ah.clear()
        idx = np.unique(np.linspace(0, T, min(T + 1, 6000), dtype=int))
        ax.plot(idx / n, x[idx] / n, lw=0.8, color="#287ab0", label="Ein Versuch")
        ax.axhline(0.5, color="grey", ls=":")
        ax.set(
            xlim=(0, dauer.value),
            ylim=(-0.02, 1.02),
            xlabel="Schritte / N",
            ylabel="Anteil k/N in Behälter 0",
        )
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(alpha=0.2)
        freq = np.bincount(x, minlength=n + 1) / (T + 1)
        q = stationaer(n)
        levels = np.arange(n + 1) / n
        ah.barh(levels, freq, height=0.9 / n, color="#287ab0", label="Blauer Versuch")
        ah.plot(q, levels, color="#c56a12", lw=1.3, label="Gleichgewicht qₖ")
        ah.axhline(0.5, color="grey", ls=":")
        ah.set(xlabel="Besuchsanteil / qₖ", title="Häufigkeiten")
        ah.grid(axis="x", alpha=0.2)
        ah.legend(fontsize=7, loc="upper right")
        zeichnen(fig, pic)
        info.value = f"{T} Schritte; Start bei k = {x[0]}. Rechts: alle {T+1} Zustände des blauen Versuchs, einschließlich Start. Ein Balken pro k."
        if T > 6000:
            info.value += " Die Verlaufslinie zeigt eine Auswahl der Zeitpunkte; das Histogramm verwendet alle."
        r["last"] = {"path": x.copy(), "histogram": freq, "q": q, "steps": T}

    def new(_):
        r["seed"] += 1
        update()

    for w in (N, start, dauer):
        w.observe(update, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                feld("N", N),
                feld("Anfangszustand", start),
                feld("Dauer in Vielfachen von N Schritten", dauer),
                btn,
                pic,
                info,
            ]
        )
    )
    update()
    PANELS["relaxation"] = r
    return r


def station_entropie():
    N = auswahl([10, 50, 200, 1000], 50)
    start = auswahl(STARTS, STARTS[0])
    t = regler(100, 0, 400)
    btn = knopf()
    pic = bild()
    info = W.HTML()
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.4))
    plt.close(fig)
    r = {
        "N": N,
        "start": start,
        "t": t,
        "button": btn,
        "bild": pic,
        "seed": 2026,
        "key": None,
        "last": None,
        "lock": False,
    }

    def update(_=None):
        if r["lock"]:
            return
        key = (N.value, start.value, r["seed"])
        if r["key"] != key:
            r["path"] = makropfade(
                N.value, 8 * N.value, startwert(N.value, start.value), r["seed"], 1
            )[0]
            r["key"] = key
        S = entropien(N.value)
        T = t.value
        x = r["path"]
        values = S[x]
        j = x[T]
        for ax in axes:
            ax.clear()
            ax.grid(alpha=0.2)
        axes[0].plot(np.arange(N.value + 1) / N.value, S / N.value)
        axes[0].scatter(j / N.value, S[j] / N.value, color="#c46813")
        axes[0].set(
            xlabel="Anteil k/N",
            ylabel="Entropie pro Molekül S(k)/N",
            title="Entropie der Makrozustände",
            ylim=(-0.02, 0.72),
            xlim=(0, 1),
        )
        axes[1].plot(
            np.arange(T + 1) / N.value, values[: T + 1] / N.value, color="#c46813", lw=1
        )
        axes[1].scatter(T / N.value, values[T] / N.value, color="#c46813")
        axes[1].set(
            xlabel="Schritte / N",
            title="Entlang eines Versuchs",
            ylim=(-0.02, 0.72),
            xlim=(0, 8),
        )
        zeichnen(fig, pic)
        delta = float(values[T] - values[T - 1]) if T else None
        info.value = f"Schritt {T}: k = {j}, S(k) = {values[T]:.4f}. " + (
            "Noch kein Übergang."
            if delta is None
            else f"Änderung seit dem vorigen Schritt: <b>{delta:+.4f}</b>."
        )
        r["last"] = {"path": x.copy(), "S": S, "values": values, "delta": delta}

    def newN(_):
        r["lock"] = True
        t.max = 8 * N.value
        t.value = 2 * N.value
        r["lock"] = False
        update()

    def new(_):
        r["seed"] += 1
        update()

    N.observe(newN, names="value")
    start.observe(update, names="value")
    t.observe(update, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                feld("N", N),
                feld("Anfangszustand", start),
                feld("Schritt: dieselbe Folge vor- und zurückspulen", t),
                btn,
                pic,
                info,
            ]
        )
    )
    update()
    PANELS["entropie"] = r
    return r


def station_rueckkehr():
    N = auswahl([4, 8, 12], 8)
    T = 30000
    btn = knopf("200 neue Versuche")
    pic = bild()
    info = W.HTML()
    fig, ax = plt.subplots(figsize=(7.4, 3.4))
    plt.close(fig)
    r = {"N": N, "T": T, "button": btn, "bild": pic, "seed": 2026, "key": None, "last": None}

    def update(_=None):
        key = (N.value, r["seed"])
        if r["key"] != key:
            r["times"] = rueckkehrzeiten(N.value, 200, T, r["seed"])
            r["key"] = key
        times = r["times"]
        cap = T
        observed = np.sort(times[times <= cap])
        n = len(observed)
        lower = np.minimum(times, cap)
        ax.clear()
        end = cap if n < 200 else max(2, int(observed.max()))
        width = max(1, math.ceil(end / 40))
        edges = np.arange(0.5, end + width + 0.5, width)
        counts, _ = np.histogram(observed, bins=edges)
        ax.bar(
            edges[:-1],
            counts / 200,
            width=np.diff(edges),
            align="edge",
            color="#287ab0",
            edgecolor="white",
            linewidth=0.4,
        )
        ax.set(
            xlim=(0.5, edges[-1]),
            xlabel="Erste positive Rückkehrzeit τ (Schritte)",
            ylabel="Anteil aller 200 Versuche\npro Zeitklasse",
            title=f"Beobachtete Rückkehrzeiten · Klassenbreite {width} Schritte",
        )
        ax.grid(axis="y", alpha=0.2)
        zeichnen(fig, pic)
        info.value = tabelle(
            ["Zurück bis T", "Bei T noch offen", "Mittel von min(τ,T)"],
            [[f"{n} / 200", 200 - n, f"{lower.mean():.2f}"]],
        )
        info.value += f"Die Balkenhöhen summieren sich zu {n/200:.1%}. Noch offene Versuche sind nicht als Rückkehr bei T eingezeichnet. Sie gehen nur in das abgeschnittene Mittel mit T ein; ihre Rückkehrzeit ist größer."
        r["last"] = {
            "times": times.copy(),
            "observed": observed,
            "open": 200 - n,
            "clipped_mean": float(lower.mean()),
            "clipped": lower,
            "histogram": counts / 200,
            "edges": edges,
        }

    def new(_):
        r["seed"] += 1
        update()

    N.observe(update, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                feld("N (jeder Versuch startet bei k = N)", N),
                W.HTML("Feste Beobachtungsgrenze: <b>T = 30 000 Schritte</b> je Versuch."),
                btn,
                pic,
                info,
            ]
        )
    )
    update()
    PANELS["rueckkehr"] = r
    return r


def station_zeitumkehr():
    N = auswahl([4, 10, 30, 100, 200], 30)
    start = auswahl(
        ["Alle in Behälter 0", "Zufälliger Gleichgewichtszustand"], "Alle in Behälter 0"
    )
    richtung = W.ToggleButtons(options=["Aufgezeichnet", "Umgekehrt"], value="Aufgezeichnet")
    t = regler(0, 0, 180)
    btn = knopf()
    pic = bild()
    info = W.HTML()
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.4))
    plt.close(fig)
    r = {
        "N": N,
        "start": start,
        "richtung": richtung,
        "t": t,
        "button": btn,
        "bild": pic,
        "seed": 2026,
        "key": None,
        "last": None,
        "lock": False,
    }

    def update(_=None):
        if r["lock"]:
            return
        key = (N.value, start.value, r["seed"])
        n = N.value
        if r["key"] != key:
            data = mikropfad(n, 6 * n, n, r["seed"])
            if start.value != "Alle in Behälter 0":
                initial = (
                    np.random.default_rng(r["seed"] + 100003)
                    .integers(0, 2, size=n)
                    .astype(bool)
                )
                data["states"] = np.logical_xor(data["states"], ~initial)
                data["k"] = data["states"].sum(axis=1)
            r["data"] = data
            r["key"] = key
        reverse = richtung.value == "Umgekehrt"
        states = r["data"]["states"][::-1] if reverse else r["data"]["states"]
        actions = r["data"]["actions"][::-1] if reverse else r["data"]["actions"]
        x = states.sum(axis=1)
        S = entropien(n)[x]
        for ax in axes:
            ax.clear()
            ax.grid(alpha=0.2)
        axes[0].plot(np.arange(len(x)) / n, x / n)
        axes[0].scatter(t.value / n, x[t.value] / n, color="#c46813")
        axes[0].set(
            xlabel="Abspielschritte / N",
            ylabel="Anteil in Behälter 0",
            ylim=(-0.02, 1.02),
            xlim=(0, 6),
        )
        axes[1].plot(np.arange(len(x)) / n, S / n, color="#c46813")
        axes[1].scatter(t.value / n, S[t.value] / n, color="#287ab0")
        axes[1].set(
            xlabel="Abspielschritte / N",
            ylabel="Entropie pro Molekül",
            ylim=(-0.02, 0.72),
            xlim=(0, 6),
        )
        zeichnen(fig, pic)
        state = states[t.value]

        def ids_text(mask):
            ids = np.flatnonzero(mask) + 1
            return (
                ", ".join(map(str, ids[:15]))
                + (f" … ({len(ids)} insgesamt)" if len(ids) > 15 else "")
            ) or "keine"

        ids0 = ids_text(state)
        ids1 = ids_text(~state)
        a = int(actions[t.value - 1]) if t.value else None
        last = (
            "Startbild."
            if a is None
            else (
                "Letzte Aktion: Pause." if a == n else f"Letzte Aktion: Molekül {a+1} wechseln."
            )
        )
        info.value = f"<b>{richtung.value}, Schritt {t.value}: k = {x[t.value]}</b><br>{last}<br>Behälter 0: {ids0}<br>Behälter 1: {ids1}<br>Filmstart: k = {x[0]}, S = {S[0]:.3f}; Filmende: k = {x[-1]}, S = {S[-1]:.3f}.<br>Beim Umkehren werden dieselben Aktionen in umgekehrter Reihenfolge abgespielt; es wird kein neuer Versuch ausgelost."
        r["last"] = {"states": states.copy(), "actions": actions.copy(), "k": x, "S": S}

    def newN(_):
        r["lock"] = True
        t.max = 6 * N.value
        t.value = 0
        r["lock"] = False
        update()

    def new(_):
        r["seed"] += 1
        update()

    N.observe(newN, names="value")
    for w in (start, richtung, t):
        w.observe(update, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                feld("N", N),
                feld("Anfang der aufgezeichneten Folge", start),
                richtung,
                feld("Abspielschritt", t),
                btn,
                pic,
                info,
            ]
        )
    )
    update()
    PANELS["zeitumkehr"] = r
    return r


print("Bereit – weiter mit Station 1.")


<br><br>

## 1 · Das Ehrenfest-Modell

Der **Mikrozustand** gibt für jedes nummerierte Molekül seinen Behälter an. Der **Makrozustand** $k$ zählt nur die Moleküle in Behälter 0.

1. **Schiebe den Schrittregler zunächst einzeln weiter.** Was kann mit $k$ passieren? Beim Zurückschieben siehst du die selbe Aufzeichnung des Experiments rückwärts. <span style="color: #707070;">(leicht)</span>
2. Wie viele Mikrozustände gehören für $N = 4$ Teilchen zum Makrozustand $k = 2$? Wie viele Mikrozustände gehören für $N = 6$ Teilchen zum Makrozustand $k = 2$? <span style="color: #707070;">(leicht)</span>
3. Begründe, warum es für beliebiges $N$ stets $\frac{N(N-1)}{2}$ Mikrozustände zum Makrozustand $k = 2$ gibt (Warum ist $\frac{N(N-1)}{2}$ überhaupt eine ganze Zahl?). <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_1 = station_teilchen()

<br><br>

## 2 · Die Übergänge selbst bestimmen

**Wie wahrscheinlich sind Abnahme, Stillstand und Zunahme des Makrozustands $k$?** 

1. Zähle jeweils die passenden Aktionen. Gib deine Ergebnisse als Brüche ein und prüfe sie. Nach einer richtigen Eingabe erscheint die vollständige Übergangsmatrix. <span style="color: #707070;">(leicht)</span>
2. Leite eine Formel für die Wahrscheinlichkeit der Ereignisse "k nimmt um 1 ab", "k bleibt gleich" und "k nimmt um 1 zu" für beliebige $N$ und $k$ her. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_2 = station_uebergaenge()

<br><br>

## 3 · Welche Zustände sind häufig?

Zu $k$ gehören $\binom Nk=\frac{N!}{k!(N-k)!}$ Mikrozustände. Sind die $2^N$ Mikrozustände gleich wahrscheinlich, gilt für den Makrozustand $q_k=\binom Nk/2^N$. Das Diagramm zeigt $q_k$ über dem Anteil $k/N$ in Behälter 0.

1. Probiere verschiedene Werte von $N$ aus. Warum wird die Wahrscheinlichkeit, dass der Anteil $k / N$ zwischen 40% und 60% liegt (dunkle Balken im Diagramm) immer größer, je größer $N$ ist? <span style="color: #707070;">(schwerer)</span>
2. Wir nehmen nun an, dass $X_t$ die Verteilung $q$ hat. Begründe, warum die Wahrscheinlichkeit, erst bei $k$ zu sein und dann nach $k+1$ zu wechseln, gegeben ist durch $P(X_t=k,\,X_{t+1}=k+1)=q_k\frac{N-k}{N+1}$. <span style="color: #707070;">(schwerer)</span>
3. Für den umgekehrten Übergang ist die Wahrscheinlichkeit gegeben durch $P(X_t=k + 1,\,X_{t+1}=k) = q_{k+1}\frac{k+1}{N+1}$. Zeige nun mithilfe der Gleichheit $\binom N{k+1}=\binom Nk\frac{N-k}{k+1},$ dass beide Wahrscheinlichkeiten gleich sind, also dass die **detailed balance**-Bedingung gilt. Begründe daraus, dass die Verteilung nach einem Schritt wieder $q$ ist, also $Mq=q$. <span style="color: #707070;">(schwer)</span>


In [ ]:
station_3 = station_gleichgewicht()

<br><br>

## 4 · Das Ehrenfest-Modell in Aktion

Links siehst du den Anteil $k/N$ entlang eines Versuchs (blau). Das **waagerechte Histogramm rechts** (blau) zählt, wie häufig die Zustände dieses Versuchs besucht wurden. Die gemeinsame Höhenachse zeigt $k/N$; die orange Vergleichskurve rechts zeigt die Gleichgewichtsverteilung aus Station 3.

1. Warum schwankt die blaue Kurve nach der Anfangsphase immer weniger um den Gleichgewichtsanteil $1/2$, je größer $N$ gewählt wird? Begründe deine Antwort mithilfe der Grafik aus Station 3. <span style="color: #707070;">(schwerer)</span>
2. Warum nähert sich das blaue Histogramm rechts immer mehr der orangen Linie an, je höher man die Anzahl der Schritte in Vielfachen von $N$ wählt? <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_4 = station_relaxation()

<br><br>

## 5 · Muss die Entropie immer steigen?

Die **Boltzmann-Entropie** des Makrozustands $k$ ist (bis auf eine physikalische Konstante) gleich $S(k)=\log\binom Nk$ (Wir verwenden hier den natürlichen Logarithmus). In der Grafik unten siehst du links $S(k)/N$ in Abhängigkeit von $k / N$ (blaue Linie). Rechts siehst du den Verlauf von $S(k) / N$ entlang eines Versuchs im Ehrenfest-Modell (orange Linie).

1. **Suche mit dem Schrittregler eine Entropieabnahme.** Die Ausgabe nennt die Änderung gegenüber dem vorigen Schritt. Beginne einmal mit allen Molekülen links und einmal mit einem etwa 50:50 verteilten Startzustand. <span style="color: #707070;">(leicht)</span>
2. Warum hat $S(k) / N$ ein Maximum genau bei $k / N = 0.5$? <span style="color: #707070;">(schwerer)</span>
3. Vergleiche N = 10, 200 und 1000. Warum sieht die Entropie für größeres N immer mehr wie eine monoton wachsende Funktion ohne Schwankungen aus? Begründe deine Antwort mithilfe der Grafik aus Station 3. <span style="color: #707070;">(schwerer)</span>
4. Formuliere, was „Entropiezunahme“ hier beschreibt, ohne zu behaupten, dass jeder einzelne Schritt die Entropie erhöht. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_5 = station_entropie()

<br><br>

## 6 · Wie lange dauert eine Rückkehr?

Jeder Versuch startet bei $k=N$, also alle Moleküle befinden sich zu Beginn in Behälter 0. Wir suchen den Zeitpunkt in der Zukunft, wenn sich erstmals wieder alle Moleküle in Behälter 0 befinden. Mathematisch wird dies beschrieben durch die **erste positive Rückkehrzeit** $\tau=\min\{t\geq1:X_t=N\}$ (auch eine Pause im ersten Ursprungszustand zählt als Rückkehr). 

Das Histogramm zeigt die in **200 Versuchen** beobachteten Rückkehrzeiten. Jeder Versuch wird bis zur Rückkehr, höchstens aber **T = 30 000 Schritte** lang beobachtet. Jede Balkenhöhe ist der Anteil **aller 200 Versuche** im jeweiligen Zeitintervall. Noch offene Versuche fehlen im Histogramm; ihre Anzahl steht darunter.

1. Die stationäre Wahrscheinlichkeit des Startzustands ist $q_N=2^{-N}$. Wie groß ist die mittlere Rückkehrzeit $E[\tau]$? Nutze den Zusammenhang zwischen stationärer Wahrscheinlichkeit und mittlerer Rückkehrzeit aus dem Skript. <span style="color: #707070;">(schwerer)</span>
2. Vergleiche **$N=4$, $N=8$ und $N=12$**. Liegen die Rückkehrzeiten meist nahe am theoretischen Mittelwert aus Teilaufgabe 1? Wähle anschließend $N=12$ und lose dreimal neue Versuche aus. Erkläre anhand der Histogramme, warum wenige sehr lange Rückkehrzeiten den Mittelwert stark beeinflussen können. Achte auch darauf, ob noch Versuche offen sind. <span style="color: #707070;">(schwerer)</span>

Die Tabelle zeigt den Mittelwert von $\min(\tau,T)$: Offene Versuche werden dort vorläufig mit $T$ gezählt. Er ist höchstens so groß wie der Mittelwert der vollständigen Rückkehrzeiten derselben 200 Versuche; auch dieser müsste nicht genau $E[\tau]$ ergeben.

<details>
<summary>Ergebnis für Teilaufgabe 2, falls du Teilaufgabe 1 überspringst</summary>

Für den Start $X_0=N$ und die erste positive Rückkehrzeit gilt
$$\mathbb E_N[\tau]=\frac{1}{q_N}=2^N.$$
Für $N=4,8,12$ sind dies **16, 256 und 4096 Schritte**. Dies sind theoretische Erwartungswerte; die Mittelwerte endlich vieler Versuche können davon abweichen. Der Beweis der Formel ist für den Vergleich nicht nötig.

</details>


In [ ]:
station_6 = station_rueckkehr()

<br><br>

## 7 · Kann man den „Film“ umkehren?

Wir spielen nun **denselben Versuch** im Ehrenfest-Modell vorwärts oder rückwärts ab. Der Makrozustand $k_t$ zählt die Moleküle in Behälter 0; $S(k)=\log\binom Nk$.

1. Wähle **$N=200$** und „Alle in Behälter 0“. Vergleiche jeweils Anfang und Ende des Films in beiden Richtungen. Welche Richtung wirkt typisch? Wähle nun „Zufälliger Gleichgewichtszustand“. Ist eine Zeitrichtung jetzt ähnlich deutlich erkennbar? <span style="color: #707070;">(leicht)</span>
2. Setze $N=4$. Berechne die Wahrscheinlichkeit des Makropfads $4\to3\to2$, wenn der Anfang $k_0=4$ feststeht, und die des umgekehrten Pfads $2\to3\to4$, wenn $k_0=2$ feststeht. Welche ist größer, und um welchen Faktor? Jeder einzelne Übergang zwischen zwei Mikrozuständen ist gleich wahrscheinlich – warum folgt daraus trotzdem nicht, dass beide Makropfade gleich wahrscheinlich sind? <span style="color: #707070;">(schwerer)</span>
3. **Vertiefung: Pfade und ihre Umkehr.** Schreibe $p_{ij}=P(X_{t+1}=j\mid X_t=i)$. Für einen zulässigen Makropfad $k_0,\ldots,k_T$ sind die Wahrscheinlichkeiten des Vorwärts-Makropfads und des Rückwärts-Makropfads bei festem Anfang gegeben durch
   $$P_{\mathrm{vor}}=\prod_{t=0}^{T-1}p_{k_tk_{t+1}},\qquad
   P_{\mathrm{rück}}=\prod_{t=0}^{T-1}p_{k_{t+1}k_t}.$$
   Leite aus der detailed balance-Bedingung $q_i p_{ij}=q_j p_{ji}$ aus Station 3 her: $\frac{P_{\mathrm{vor}}}{P_{\mathrm{rück}}}
   =\frac{q_{k_T}}{q_{k_0}}
   =\exp\big(S(k_T)-S(k_0)\big).$ Überprüfe diese Formel anhand von Teilaufgabe 2. Was sagt die Formel intuitiv aus? <span style="color: #707070;">(schwer)</span>


In [ ]:
station_7 = station_zeitumkehr()

<br><br>

## Kleine Codeaufgabe · Mittlere Änderung (optional) <span style="color: #707070;">(schwerer)</span>

In einem Schritt ändert sich $k$ um $-1$, $0$ oder $+1$. **Ersetze `None` durch den gewichteten Mittelwert dieser drei Änderungen.** Die Wahrscheinlichkeiten sind bereits eingetragen.

Probiere anschließend Zustände unterhalb, oberhalb und genau bei $N/2$. Was sagt das Vorzeichen über die erwartete Bewegung aus? Warum muss der nächste einzelne Schritt nicht in diese Richtung gehen?

In [ ]:
N = 20
k = 15  # Wähle eine ganze Zahl zwischen 0 und N.
p_ab = k / (N + 1)
p_pause = 1 / (N + 1)
p_auf = (N - k) / (N + 1)

mittlere_aenderung = None  # Ergänze den gewichteten Mittelwert.

if mittlere_aenderung is None:
    print("Ersetze None durch deine Rechnung.")
else:
    print("Erwartete Änderung von k in einem Schritt:", mittlere_aenderung)
    print("Deutung: positiv = im Mittel Zunahme, negativ = Abnahme.")